# Reasoning and checking model answers

You will build a small Python program that decides whether proposed answers deserve to be accepted. You will extract a number from text, count repeated answers, and check a result against a calculation you control. You need only the variables, lists, dictionaries, functions, loops, and conditionals from your first Python course.

This lesson is self-contained. If you completed Chapter 2, remember its pattern: create messages, send a request, receive text, and record the response. Here you add a decision between receiving that text and presenting an accepted answer.

Steps 1 through 8 need no downloaded model, dataset, GPU, CUDA, or paid API. Read each section before running its code cell, predict the output, and compare it with the checkpoint. If you see a NameError, rerun the earlier cells in order.


## Before you code

A language model generates text using learned numerical parameters called weights. Training changes those parameters. Inference produces output using the parameters already in place. Editing a prompt changes the request, not those parameters.

A prompt without a worked demonstration is called zero-shot. Adding one example question and its supplied answer makes it one-shot. Sampling means generating candidate responses using the model's token probabilities. Temperature changes how concentrated that sampling is.

Self-consistency commonly samples several solutions and aggregates their final answers. A verifier checks requirements or evidence separately. Repeated answers show agreement, while a suitable independent check can establish something about correctness. A plausible explanation does not establish that its steps caused the answer.


## Step 1: Model the inventory changes

You manage a shared workshop tool room. It starts with 18 drills available. Students check out 5 drills, and another group returns 3 usable drills. How many drills are now available? Work it out on paper first: checking out reduces the count, and returning increases it.

The variable `problem` holds a string. The other variables hold integers. A single equals sign assigns a value. Python calculates the right side before assigning it to `expected`. `expected` is your independently calculated reference, not an answer obtained from a model.


In [1]:
problem = "18 drills in stock. 5 checked out, 3 returned. How many available?"

start = 18
checked_out = 5
returned = 3
expected = start - checked_out + returned

print(problem)
print("Reference count:", expected)


18 drills in stock. 5 checked out, 3 returned. How many available?
Reference count: 16


## Step 2: Construct three requests

Square brackets create a list, which keeps its items in order. Braces create a dictionary, which associates keys with values. Each message has the keys `role` and `content`. `user` labels a request. `assistant` labels the example answer you supply yourself.

`direct` holds one message. `brief` also holds one message but asks for a short calculation. Both are zero-shot because neither supplies a worked demonstration. `worked` holds an example user message, an example assistant response, and the new task. That makes it one-shot. The example uses a different inventory item and different numbers so the model has to apply the pattern. Creating these lists makes no network call and trains nothing.


In [2]:
format_rule = "Use a final line in this format: Answer: <integer>"

direct = [{"role": "user", "content": problem + " " + format_rule}]

brief_text = problem + " Show a short calculation. " + format_rule
brief = [{"role": "user", "content": brief_text}]

worked = [
    {"role": "user",
     "content": "10 cords in stock. 4 borrowed, 2 returned. How many available?"},
    {"role": "assistant", "content": "10 - 4 = 6, then 6 + 2 = 8.\nAnswer: 8"},
    {"role": "user", "content": problem + " Show a short calculation. " + format_rule},
]

print("Direct message count:", len(direct))
print("Worked message count:", len(worked))
print(worked[1]["content"])


Direct message count: 1
Worked message count: 3
10 - 4 = 6, then 6 + 2 = 8.
Answer: 8


## Step 3: Supply controlled test responses

You will first test your Python without involving a model. A fixture is a predetermined input used to exercise software. These five fixtures deliberately include repeated mistakes and a missing final-answer label. They cannot tell you how often any real model succeeds.

The first, second, and fourth fixtures omit the returned drills. The third includes both events. The fifth gives the right quantity in words but breaks the requested format. This lets you separate a formatting failure from an incorrect number.


In [3]:
scripted = [
    "18 - 5 = 13.\nAnswer: 13",
    "Only the checkout matters.\nAnswer: 13",
    "18 - 5 + 3 = 16.\nAnswer: 16",
    "I counted thirteen available.\nAnswer: 13",
    "Sixteen drills are available.",
]

for text in scripted:
    print(repr(text))

print("Source: SCRIPTED TEST FIXTURES")


'18 - 5 = 13.\nAnswer: 13'
'Only the checkout matters.\nAnswer: 13'
'18 - 5 + 3 = 16.\nAnswer: 16'
'I counted thirteen available.\nAnswer: 13'
'Sixteen drills are available.'
Source: SCRIPTED TEST FIXTURES


## Step 4: Convert final answer text to an integer

A parser converts text in a known format into a value your program can use. `def` introduces a function; `text` is its parameter. `strip` removes whitespace at the beginning and end, and `splitlines` separates the response into lines.

An empty list counts as false, so `if not lines` handles an empty response before you try to index it. `lines[-1]` selects the last line. `startswith` checks the required label, and the slice `[len(prefix):]` keeps everything after it. `int` attempts the conversion, and `except ValueError` handles the failure. `None` means no usable answer, and differs from the valid integer zero. Never use `eval` or `exec` to process a model response.


In [4]:
def parse_answer(text):
    lines = text.strip().splitlines()
    if not lines:
        return None
    final_line = lines[-1].strip()
    prefix = "Answer:"
    if not final_line.startswith(prefix):
        return None
    number_text = final_line[len(prefix):].strip()
    try:
        return int(number_text)
    except ValueError:
        return None

answers = []
for text in scripted:
    answers.append(parse_answer(text))

print(answers)


[13, 13, 16, 13, None]


## Step 5: Count the votes

`counts` is a dictionary from an answer to its frequency. The explicit `is None` check skips missing values while keeping zero. `continue` moves directly to the next loop iteration. For a previously unseen answer, create a counter at zero, then add one.

After counting, an empty dictionary means there are no valid answers. `max` finds the largest frequency, and the second loop collects every answer with that frequency. If there is more than one leader, report a tie. Otherwise the single most frequent answer wins. This rule chooses a plurality, which does not always exceed half the valid votes.


In [5]:
def vote_answers(values):
    counts = {}
    for value in values:
        if value is None:
            continue
        if value not in counts:
            counts[value] = 0
        counts[value] = counts[value] + 1
    if not counts:
        return {"answer": None, "counts": counts, "status": "no valid answers"}
    highest_count = max(counts.values())
    leaders = []
    for value in counts:
        if counts[value] == highest_count:
            leaders.append(value)
    if len(leaders) > 1:
        return {"answer": None, "counts": counts, "status": "tie"}
    return {"answer": leaders[0], "counts": counts, "status": "selected"}

vote = vote_answers(answers)
print(vote)
print("Matches reference:", vote["answer"] == expected)


{'answer': 13, 'counts': {13: 3, 16: 1}, 'status': 'selected'}
Matches reference: False


## Step 6: Check the final outcome

Your verifier uses a rule whose answer came from the inventory facts. It parses a candidate and compares that value with `expected_answer`, a second function parameter. Using a parameter lets you reuse the function with a different problem later.

A format error and a wrong integer both earn score 0, but their statuses remain different. A match earns score 1. This binary score describes this one check only. Each returned record keeps the raw text so you can examine what was actually said. This is a rule-based outcome check, not a trained reward model.


In [6]:
def score_candidate(text, expected_answer):
    answer = parse_answer(text)
    if answer is None:
        status = "format error"
    elif answer == expected_answer:
        status = "correct"
    else:
        status = "wrong answer"
    score = 0
    if status == "correct":
        score = 1
    return {"text": text, "answer": answer, "score": score, "status": status}

records = []
for text in scripted:
    records.append(score_candidate(text, expected))

for record in records:
    print(record["answer"], record["score"], record["status"])


13 0 wrong answer
13 0 wrong answer
16 1 correct
13 0 wrong answer
None 0 format error


## Step 7: Select by score

Best-of-N means choosing from N candidates using a scoring rule. Here N is five, including the malformed candidate. Your scores are only 0 or 1, so every passing candidate ties for the best possible score. You will explicitly keep the first passing candidate.

The function examines each record in order. `return` immediately ends the search when a score equals 1. If the loop finishes without a match, the last `return` supplies `None`. The caller tests that case before attempting to read a dictionary key. This simple selector does not rank explanations, repair responses, or create a missing answer.


In [7]:
def select_best(records):
    for record in records:
        if record["score"] == 1:
            return record
    return None

best = select_best(records)
if best is None:
    print("No candidate passed.")
else:
    print("Selected count:", best["answer"])
    print("Original response:", best["text"])

print("Pool size:", len(records))


Selected count: 16
Original response: 18 - 5 + 3 = 16.
Answer: 16
Pool size: 5


## Step 8: Inspect the displayed intermediate states

A state is the count at one point: before checkout, after checkout, and after returns. The list `[18, 13, 16]` follows the facts. The list `[18, 14, 16]` ends at the right count even though both changes are wrong.

The length test prevents an `IndexError` when a list has missing positions. Each `==` expression produces a Boolean. `states[1]` should equal `states[0]` minus the checked-out count. `states[2]` should equal `states[1]` plus the return count. Notice that each transition uses the previous reported state, so an incorrect state can cause a later transition to fail too. A final-answer check sees only the last number. A process check can identify invalid displayed transitions.


In [8]:
def check_process(states, start_count, out_count, return_count):
    if len(states) != 3:
        return [False, False, False]
    return [
        states[0] == start_count,
        states[1] == states[0] - out_count,
        states[2] == states[1] + return_count,
    ]

correct_steps = [18, 13, 16]
lucky_final = [18, 14, 16]

print("Valid states:", check_process(correct_steps, start, checked_out, returned))
print("Invalid transitions:", check_process(lucky_final, start, checked_out, returned))
print("Same final value:", correct_steps[-1] == lucky_final[-1] == expected)


Valid states: [True, True, True]
Invalid transitions: [True, False, False]
Same final value: True


## When you do not know the answer yet

The reference value in this lab makes the Python easy to inspect. A production application often receives a new question without a prewritten answer. You then need trustworthy facts, calculations, or requirements that are independent of the candidate being judged.

- **Inventory lookup:** Look up the current inventory and transaction records. Let Python calculate the available count from those records, then compare the model's explanation with that computed value. The records might still be stale or incomplete, so keep their timestamp and ask for clarification when transactions conflict.
- **Sort properties:** For a request to sort inventory records, check properties without prewriting the sorted list: quantities must be in nondecreasing order, and every original record must remain exactly once. Check duplicates as well as total length.
- **Policy passages:** For a policy question, retrieve the actual policy passages and have the answer identify the evidence for each claim. Check whether each passage supports the claim, whether it is current, and whether an exception applies. Missing evidence should lead to a question, a qualified answer, or a refusal to select a candidate.
- **Design decisions:** For a design decision such as which tool to buy, several answers may fit different priorities. Compare requirements, assumptions, and evidence rather than pretending to have a single mathematical verifier. No extra votes can settle an unstated preference.


## Optional extension: install the local model

Complete the offline steps first. Ollama is a separate application that runs a model on your computer and accepts requests from Python. The model tag `gemma3:1b` selects the small model used here. Your computer needs room for the application and model download. CPU operation can be slow, and a GPU is optional for this exercise. No Python package in requirements.txt installs Ollama.

On Windows, download the installer from https://ollama.com/download/windows and run it. Launch Ollama, then open a new PowerShell terminal and enter these commands, one at a time:

- `ollama --version`
- `ollama pull gemma3:1b`
- `ollama list`
- `Invoke-RestMethod http://localhost:11434/api/tags`
- `ollama run gemma3:1b "What is 2 + 2?"`

Confirm the exact model tag appears in the model list and the tags endpoint responds. `localhost` means the same computer as Python, and `11434` is the server's port. Run this optional exercise in local VS Code. Record failures as failures rather than substituting fixtures.


## Optional step 9: Define the request function

The imports load Python standard-library modules. `json` converts dictionaries to portable JSON text and back. `time` measures duration. `urllib.request` sends HTTP requests. `MODEL` and `URL` hold configuration values. Capitals are a naming convention, not special Python syntax.

Inside `ask_model`, `payload` holds the request settings. `json.dumps` makes JSON text, and `encode` makes transport bytes. `Request` combines those bytes with the destination and a header identifying the content as JSON. Only `urlopen` sends the request, when you call the function. The `with` block closes the reply automatically. An empty choices list raises an explicit error. The 180-second timeout applies to the HTTP operation, not the whole notebook. Defining this function makes no request.


In [9]:
import json
import time
import urllib.request

MODEL = "gemma3:1b"
URL = "http://localhost:11434/v1/chat/completions"

def ask_model(messages, temperature=0.0):
    payload = {
        "model": MODEL, "messages": messages,
        "temperature": temperature, "max_tokens": 180, "stream": False,
    }
    request = urllib.request.Request(
        URL, data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=180) as reply:
        data = json.loads(reply.read().decode("utf-8"))
    choices = data.get("choices", [])
    if not choices:
        raise ValueError("The server returned no completion choices")
    return {
        "text": choices[0]["message"].get("content") or "",
        "model": data.get("model", MODEL),
        "usage": data.get("usage", {}),
        "seconds": time.perf_counter() - started,
    }


## Optional step 10: Compare prompts

Keep `RUN_LIVE` False until your local server and model checks pass. An `if` statement controls whether the requests run. The list of pairs associates a style label with each message list. Each call receives the same model configuration and temperature, while the input text changes.

Each response passes through the same `score_candidate` function as the fixtures. With the switch enabled, this cell makes three sequential requests. One success does not show that a style is generally better. Keep the raw results, including wrong numbers and formatting failures, and describe only what you observed.


In [10]:
RUN_LIVE = False

live_comparisons = []

if RUN_LIVE:
    styles = [("direct", direct), ("one-shot", worked), ("brief calculation", brief)]
    for label, messages in styles:
        response = ask_model(messages, temperature=0.0)
        scored = score_candidate(response["text"], expected)
        comparison = {"style": label, "response": response, "scored": scored}
        live_comparisons.append(comparison)
        print(label, scored["status"], round(response["seconds"], 2))
        print(response["text"])
        print("Usage:", response["usage"])
else:
    print("Prompt comparison skipped; no request sent.")


Prompt comparison skipped; no request sent.


## Optional step 11: Sample and select

`RUN_SAMPLING` is independent of `RUN_LIVE`. Keep it False for the offline run. `N` sets the number of requests to five. Each request starts with the same brief prompt and no earlier candidate in its context.

Temperature 0.8 allows sampling variation but guarantees neither distinct responses nor correctness. Each response is scored and retained. The next loop collects the parsed answers and score records, then applies voting and verifier selection to that very same pool. This makes their difference interpretable. If both switches are True, the notebook makes eight requests. Keep failures and ties visible; do not edit returned text to make a result pass.


In [11]:
RUN_SAMPLING = False
N = 5

live_samples = []

if RUN_SAMPLING:
    for sample_number in range(N):
        response = ask_model(brief, temperature=0.8)
        scored = score_candidate(response["text"], expected)
        live_samples.append({"response": response, "scored": scored})
        print(sample_number + 1, scored["answer"], scored["status"])

    live_answers = []
    live_records = []
    for sample in live_samples:
        live_answers.append(sample["scored"]["answer"])
        live_records.append(sample["scored"])

    print("Vote:", vote_answers(live_answers))
    winner = select_best(live_records)
    if winner is None:
        print("No live candidate passed.")
    else:
        print("Verifier selection:", winner["answer"])
else:
    print("Sampling skipped; no request sent.")


Sampling skipped; no request sent.


## Practice and review

Before running more code, predict `vote_answers([16, 13])`, `vote_answers([None])`, and `vote_answers([0, 0, None])`. You should get a tie, no valid answers, and a selected zero. Then test your predictions in the next cell.


In [12]:
print(vote_answers([16, 13]))
print(vote_answers([None]))
print(vote_answers([0, 0, None]))


{'answer': None, 'counts': {16: 1, 13: 1}, 'status': 'tie'}
{'answer': None, 'counts': {}, 'status': 'no valid answers'}
{'answer': 0, 'counts': {0: 2}, 'status': 'selected'}


Check `parse_answer` with empty text, `Answer: 16`, `Answer: 16 drills`, and a multiline string ending with `Answer: 16`. The empty text has no final line to read, and `Answer: 16 drills` has the label but is not an integer literal, so both fail on formatting rather than on the number itself.


In [13]:
print(parse_answer(""))
print(parse_answer("Answer: 16"))
print(parse_answer("Answer: 16 drills"))
print(parse_answer("18 - 5 + 3 = 16.\nAnswer: 16"))


None
16
None
16


## Changed problem

Change the inventory problem to 24 drills, 6 checked out, and 2 returned. Calculate 20 independently before creating new fixtures. Include at least one correct answer and two wrong answers. Rebuild the answers and records using the changed expected value.


In [14]:
start2 = 24
checked_out2 = 6
returned2 = 2
expected2 = start2 - checked_out2 + returned2
print("New reference count:", expected2)

scripted2 = [
    "24 - 6 + 2 = 20.\nAnswer: 20",
    "24 - 6 = 18.\nAnswer: 18",
    "I counted nineteen available.\nAnswer: 19",
]

answers2 = []
for text in scripted2:
    answers2.append(parse_answer(text))
print(answers2)

records2 = []
for text in scripted2:
    records2.append(score_candidate(text, expected2))
for record in records2:
    print(record["answer"], record["score"], record["status"])

best2 = select_best(records2)
if best2 is None:
    print("No candidate passed.")
else:
    print("Selected count:", best2["answer"])


New reference count: 20
[20, 18, 19]
20 1 correct
18 0 wrong answer
19 0 wrong answer
Selected count: 20


Remove every passing fixture and check that selection returns `None`. This also checks that your program is not fixed at 16.


In [15]:
failing_only = []
for text in scripted2:
    if score_candidate(text, expected2)["score"] == 0:
        failing_only.append(text)

failing_records = []
for text in failing_only:
    failing_records.append(score_candidate(text, expected2))

print("Selection with no passing fixture:", select_best(failing_records))


Selection with no passing fixture: None


## Process practice

For process practice, use `[24, 18, 20]` and `[24, 19, 20]`. Predict every Boolean before calling `check_process` with 24, 6, and 2. The first list passes all checks. The second has the right final count but invalid transitions.


In [16]:
print(check_process([24, 18, 20], 24, 6, 2))
print(check_process([24, 19, 20], 24, 6, 2))


[True, True, True]
[True, False, False]


## Explaining your evidence

**Which inputs were authored, and which came from a model?** Every input in this notebook was authored by hand: the inventory numbers, the three message lists, all five scripted fixtures, the practice values, and the changed problem. No model calls ran. Both live switches stayed `False` (`RUN_LIVE` and `RUN_SAMPLING`), so `ask_model` was defined but never called, and no request was sent. The fixtures are test inputs written to exercise the code, not measured model behavior.

**One dictionary lookup:** `worked[1]["content"]` uses two lookups. `worked[1]` selects the second dictionary in the list (Python counts from zero), and `["content"]` retrieves that dictionary's text. In Step 6, `record["score"]` looks up the stored score when selecting the best candidate.

**One loop:** The loop `for text in scripted:` takes one fixture string at a time, assigns it to `text`, and calls `parse_answer` on it, appending each result to `answers`. The loop body runs once per fixture, so five fixtures produce five parsed answers.

**One return statement:** In `parse_answer`, `return None` ends the function immediately when the last line lacks the `Answer:` label. Nothing after that line runs for that call, and the caller receives `None` to mean "no usable answer," which is different from the valid integer zero.


## Exit ticket

**Training versus inference:** Training changes a model's weights using examples or reward feedback. Inference produces output using the weights already in place. Editing a prompt changes the request, not the weights, so asking an ordinary model for a calculation does not turn it into a newly trained reasoning model.

**Zero-shot versus one-shot:** A zero-shot prompt gives no worked demonstration. A one-shot prompt adds one example question and its supplied answer, as in `worked`, so the model can apply the pattern to the new task.

**Voting versus verification:** Voting aggregates final answers and shows agreement; the most frequent answer can still be wrong, as the vote for 13 showed. Verification checks a candidate against independent evidence or requirements, as `score_candidate` does with the independently calculated reference, and can select 16 from the same pool.

**Process versus outcome checks:** An outcome check looks only at the final number. A process check inspects the displayed intermediate states and can catch invalid transitions even when the final value happens to be right, as `[18, 14, 16]` showed.

**Where a verifier fits in a Chapter 2 program:** After receiving the raw response and before presenting an accepted answer, while retaining the raw response for inspection. Repeated candidate requests occur before selection.
